In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

# ============================================================
# CONFIGURATION
# ============================================================

# ORIGINAL ZIP
ORIGINAL_ZIP = Path(
    "/content/drive/MyDrive/BookSpineDetection.zip"
)

# Temporary extraction directory
EXTRACT_DIR = Path(
    "/content/book_dataset_original"
)

# Clean dataset directory
CLEAN_DIR = Path(
    "/content/book_dataset_clean"
)

# Final clean ZIP
CLEAN_ZIP = Path(
    "/content/BookSpineDetection_Kaggle.zip"
)

print("Original ZIP:")
print(ORIGINAL_ZIP)

print("\nClean ZIP will be:")
print(CLEAN_ZIP)

Original ZIP:
/content/drive/MyDrive/BookSpineDetection.zip

Clean ZIP will be:
/content/BookSpineDetection_Kaggle.zip


In [ ]:
# ============================================================
# EXTRACT ORIGINAL DATASET
# ============================================================

if not ORIGINAL_ZIP.exists():

    raise FileNotFoundError(
        f"Original ZIP not found:\n{ORIGINAL_ZIP}"
    )


# Remove old extraction directory
if EXTRACT_DIR.exists():

    shutil.rmtree(
        EXTRACT_DIR
    )


EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Extracting original ZIP...")

with zipfile.ZipFile(
    ORIGINAL_ZIP,
    "r"
) as z:

    z.extractall(
        EXTRACT_DIR
    )


print("Extraction completed.")

Extracting original ZIP...
Extraction completed.


In [ ]:
# ============================================================
# FIND DATASET ROOT
# ============================================================

def find_dataset_root(
    base_dir: Path
):

    required = [
        Path("train/images"),
        Path("train/labels"),

        Path("valid/images"),
        Path("valid/labels"),

        Path("test/images"),
        Path("test/labels"),
    ]


    candidates = [
        base_dir
    ]


    candidates.extend(
        p
        for p in base_dir.rglob("*")
        if p.is_dir()
    )


    for candidate in candidates:

        if all(
            (
                candidate / path
            ).exists()

            for path in required
        ):

            return candidate


    raise FileNotFoundError(
        "Could not find YOLO dataset root."
    )


DATASET_ROOT = find_dataset_root(
    EXTRACT_DIR
)


print(
    "Dataset root:"
)

print(
    DATASET_ROOT
)

Dataset root:
/content/book_dataset_original


In [ ]:
# ============================================================
# CREATE CLEAN DATASET DIRECTORY
# ============================================================

if CLEAN_DIR.exists():

    shutil.rmtree(
        CLEAN_DIR
    )


CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for split in [
    "train",
    "valid",
    "test"
]:

    (
        CLEAN_DIR
        / split
        / "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    (
        CLEAN_DIR
        / split
        / "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "Clean dataset directories created."
)

Clean dataset directories created.


In [ ]:
# ============================================================
# RENAME IMAGES + LABELS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}


def get_images(
    directory: Path
):

    return sorted(
        [
            p
            for p in directory.iterdir()

            if (
                p.is_file()
                and p.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        ]
    )


total_images = 0
total_labels = 0


for split in [
    "train",
    "valid",
    "test"
]:

    source_images = (
        DATASET_ROOT
        / split
        / "images"
    )

    source_labels = (
        DATASET_ROOT
        / split
        / "labels"
    )


    destination_images = (
        CLEAN_DIR
        / split
        / "images"
    )

    destination_labels = (
        CLEAN_DIR
        / split
        / "labels"
    )


    images = get_images(
        source_images
    )


    print(
        f"\nProcessing {split}: "
        f"{len(images)} images"
    )


    for index, image_path in enumerate(
        images,
        start=1
    ):

        # ----------------------------------------------------
        # NEW SHORT NAME
        # ----------------------------------------------------

        new_stem = (
            f"book_{index:06d}"
        )


        new_image_name = (
            new_stem
            + image_path.suffix.lower()
        )


        new_image_path = (
            destination_images
            / new_image_name
        )


        # ----------------------------------------------------
        # ORIGINAL LABEL
        # ----------------------------------------------------

        original_label = (
            source_labels
            / f"{image_path.stem}.txt"
        )


        # ----------------------------------------------------
        # NEW LABEL
        # ----------------------------------------------------

        new_label_path = (
            destination_labels
            / f"{new_stem}.txt"
        )


        # ----------------------------------------------------
        # COPY IMAGE
        # ----------------------------------------------------

        shutil.copy2(
            image_path,
            new_image_path
        )


        # ----------------------------------------------------
        # COPY LABEL
        # ----------------------------------------------------

        if original_label.exists():

            shutil.copy2(
                original_label,
                new_label_path
            )

            total_labels += 1

        else:

            # Create empty label file only if your
            # dataset legitimately has images without objects.
            new_label_path.touch()


        total_images += 1


    print(
        f"{split}: completed"
    )


print(
    "\n======================================"
)

print(
    f"Total images: {total_images}"
)

print(
    f"Total labels: {total_labels}"
)

print(
    "======================================"
)


Processing train: 5495 images
train: completed

Processing valid: 1069 images
valid: completed

Processing test: 499 images
test: completed

Total images: 7063
Total labels: 7063


In [ ]:
# ============================================================
# VERIFY CLEAN DATASET
# ============================================================

for split in [
    "train",
    "valid",
    "test"
]:

    images_dir = (
        CLEAN_DIR
        / split
        / "images"
    )

    labels_dir = (
        CLEAN_DIR
        / split
        / "labels"
    )


    images = get_images(
        images_dir
    )

    labels = list(
        labels_dir.glob("*.txt")
    )


    image_stems = {
        p.stem
        for p in images
    }

    label_stems = {
        p.stem
        for p in labels
    }


    missing_labels = (
        image_stems -
        label_stems
    )

    missing_images = (
        label_stems -
        image_stems
    )


    print(
        f"\n{split.upper()}"
    )

    print(
        "Images:",
        len(images)
    )

    print(
        "Labels:",
        len(labels)
    )

    print(
        "Missing labels:",
        len(missing_labels)
    )

    print(
        "Missing images:",
        len(missing_images)
    )


    if missing_labels:

        print(
            "Example missing labels:",
            list(missing_labels)[:5]
        )


    if missing_images:

        print(
            "Example missing images:",
            list(missing_images)[:5]
        )


    if (
        missing_labels
        or missing_images
    ):

        raise RuntimeError(
            f"Image/label mismatch detected in {split}"
        )


print(
    "\nAll image/label pairs verified successfully."
)


TRAIN
Images: 5495
Labels: 5495
Missing labels: 0
Missing images: 0

VALID
Images: 1069
Labels: 1069
Missing labels: 0
Missing images: 0

TEST
Images: 499
Labels: 499
Missing labels: 0
Missing images: 0

All image/label pairs verified successfully.


In [ ]:
# ============================================================
# CHECK FILENAME LENGTHS
# ============================================================

longest = 0
longest_path = None


for path in CLEAN_DIR.rglob("*"):

    if path.is_file():

        length = len(
            str(path).encode("utf-8")
        )


        if length > longest:

            longest = length
            longest_path = path


print(
    "Longest path length:"
)

print(
    longest,
    "bytes"
)

print(
    "\nLongest path:"
)

print(
    longest_path
)

Longest path length:
56 bytes

Longest path:
/content/book_dataset_clean/valid/images/book_000261.jpg


In [ ]:
# ============================================================
# CREATE CLEAN ZIP
# ============================================================

if CLEAN_ZIP.exists():

    CLEAN_ZIP.unlink()


print(
    "Creating clean ZIP..."
)


shutil.make_archive(
    str(
        CLEAN_ZIP.with_suffix("")
    ),
    "zip",
    root_dir=CLEAN_DIR
)


print(
    "\nClean ZIP created:"
)

print(
    CLEAN_ZIP
)


print(
    f"\nZIP size: "
    f"{CLEAN_ZIP.stat().st_size / (1024**3):.2f} GB"
)

Creating clean ZIP...

Clean ZIP created:
/content/BookSpineDetection_Kaggle.zip

ZIP size: 0.58 GB


In [ ]:
# ============================================================
# VERIFY ZIP
# ============================================================

with zipfile.ZipFile(
    CLEAN_ZIP,
    "r"
) as z:

    names = z.namelist()

    print(
        f"Files in ZIP: {len(names)}"
    )


    max_length = 0
    longest_name = None


    for name in names:

        length = len(
            name.encode("utf-8")
        )


        if length > max_length:

            max_length = length
            longest_name = name


print(
    "\nLongest ZIP entry:"
)

print(
    longest_name
)

print(
    "\nLongest entry length:"
)

print(
    max_length,
    "bytes"
)

Files in ZIP: 14135

Longest ZIP entry:
train/images/book_003623.jpg

Longest entry length:
28 bytes


In [ ]:
# ============================================================
# SHOW SAMPLE ZIP FILES
# ============================================================

with zipfile.ZipFile(
    CLEAN_ZIP,
    "r"
) as z:

    names = z.namelist()


print(
    "First 30 files:"
)


for name in names[:30]:

    print(
        name
    )

First 30 files:
test/
train/
valid/
train/images/
train/labels/
train/images/book_003623.jpg
train/images/book_002857.jpg
train/images/book_003652.jpg
train/images/book_004944.jpg
train/images/book_000261.jpg
train/images/book_002612.jpg
train/images/book_000018.jpg
train/images/book_000159.jpg
train/images/book_002138.jpg
train/images/book_003808.jpg
train/images/book_001708.jpg
train/images/book_003803.jpg
train/images/book_001731.jpg
train/images/book_001514.jpg
train/images/book_002103.jpg
train/images/book_004009.jpg
train/images/book_004008.jpg
train/images/book_003857.jpg
train/images/book_000483.jpg
train/images/book_002702.jpg
train/images/book_001117.jpg
train/images/book_001062.jpg
train/images/book_000914.jpg
train/images/book_001864.jpg
train/images/book_004873.jpg
